In [ ]:
from surprise import SVD, Dataset, Reader
import pandas as pd
import pickle

# use only users with 50+ ratings
ratings = pd.read_csv('data/processed/ratings_25m_clean.csv')
user_counts = ratings.groupby('userId').size()
active_users = user_counts[user_counts >= 50].index
ratings_small = ratings[ratings['userId'].isin(active_users)]

print(f"Training on {len(ratings_small):,} ratings")

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings_small[['userId', 'movieId', 'rating']], reader)
trainset = data.build_full_trainset()

svd = SVD(n_factors=50, n_epochs=20, random_state=42)
svd.fit(trainset)

with open('data/processed/svd_small.pkl', 'wb') as f:
    pickle.dump(svd, f)

import os
print(f"Model size: {os.path.getsize('data/processed/svd_small.pkl') / 1024 / 1024:.1f} MB")

Training on 23,107,403 ratings
Model size: 694.5 MB


In [2]:
user_counts = ratings.groupby('userId').size()
active_users = user_counts[user_counts >= 500].index
ratings_small = ratings[ratings['userId'].isin(active_users)]

# also limit to top 10000 most rated movies
top_movies = ratings.groupby('movieId').size().sort_values(ascending=False).head(10000).index
ratings_small = ratings_small[ratings_small['movieId'].isin(top_movies)]

print(f"Training on {len(ratings_small):,} ratings")
print(f"Unique users:  {ratings_small['userId'].nunique():,}")
print(f"Unique movies: {ratings_small['movieId'].nunique():,}")

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings_small[['userId', 'movieId', 'rating']], reader)
trainset = data.build_full_trainset()

svd = SVD(n_factors=50, n_epochs=20, random_state=42)
svd.fit(trainset)

with open('data/processed/svd_small.pkl', 'wb') as f:
    pickle.dump(svd, f)

import os
print(f"Model size: {os.path.getsize('data/processed/svd_small.pkl') / 1024 / 1024:.1f} MB")

Training on 8,510,638 ratings
Unique users:  9,713
Unique movies: 10,000
Model size: 234.0 MB


In [3]:
user_counts = ratings.groupby('userId').size()
active_users = user_counts[user_counts >= 1000].index
ratings_small = ratings[ratings['userId'].isin(active_users)]

top_movies = ratings.groupby('movieId').size().sort_values(ascending=False).head(5000).index
ratings_small = ratings_small[ratings_small['movieId'].isin(top_movies)]

print(f"Training on {len(ratings_small):,} ratings")
print(f"Unique users:  {ratings_small['userId'].nunique():,}")
print(f"Unique movies: {ratings_small['movieId'].nunique():,}")

reader = Reader(rating_scale=(0.5, 5.0))
data = Dataset.load_from_df(ratings_small[['userId', 'movieId', 'rating']], reader)
trainset = data.build_full_trainset()

svd = SVD(n_factors=50, n_epochs=20, random_state=42)
svd.fit(trainset)

with open('data/processed/svd_small.pkl', 'wb') as f:
    pickle.dump(svd, f)

import os
print(f"Model size: {os.path.getsize('data/processed/svd_small.pkl') / 1024 / 1024:.1f} MB")

Training on 3,307,376 ratings
Unique users:  2,675
Unique movies: 5,000
Model size: 90.8 MB


In [4]:
ratings_small.to_parquet('data/processed/ratings_small.parquet', index=False)

import os
print(f"Parquet size: {os.path.getsize('data/processed/ratings_small.parquet') / 1024 / 1024:.1f} MB")

Parquet size: 39.7 MB


In [1]:
import os
print(f"tags.csv: {os.path.getsize('data/raw/ml-25m/tags.csv') / 1024 / 1024:.1f} MB")

tags.csv: 37.0 MB


In [2]:
import pandas as pd
import pickle
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

movies = pd.read_parquet('data/processed/movies_enriched.parquet')
tags = pd.read_csv('data/raw/ml-25m/tags.csv')

tags_clean = (tags.dropna(subset=['tag'])
              .groupby('movieId')['tag']
              .apply(lambda x: ' '.join(x))
              .reset_index())
tags_clean.columns = ['movieId', 'tags']
movies = movies.merge(tags_clean, on='movieId', how='left')
movies['tags'] = movies['tags'].fillna('')
movies['genres_clean'] = movies['genres'].str.replace('|', ' ', regex=False)
movies['features'] = movies['genres_clean'] + ' ' + movies['tags']

tfidf = TfidfVectorizer(stop_words='english')
tfidf_matrix = tfidf.fit_transform(movies['features'])
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

with open('data/processed/cosine_sim.pkl', 'wb') as f:
    pickle.dump(cosine_sim, f)

import os
print(f"cosine_sim size: {os.path.getsize('data/processed/cosine_sim.pkl') / 1024 / 1024:.1f} MB")

cosine_sim size: 7.6 MB
